# 08a. NLR M3 residual 및 OU innovation 진단

M3 정상상태식으로 설명되지 않는 시간변동을 분리하고, R1 white noise, R2 OU, R3 Jump-OU 중 validation 비교에 남길 후보를 train condition만으로 선별한다.

이 단계에서는 MAD 임계값을 사용하지 않는다. 3-component Gaussian mixture는 OU innovation에 signed tail component가 존재하는지 확인하기 위한 진단 도구이며, 최종 Jump-OU 보정법은 아니다.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT = PROJECT_ROOT / 'outputs' / 'residual_diagnostics'
PROJECT_ROOT, OUTPUT

## 1. 전체 condition 진단 실행

M3는 train condition에서만 다시 적합한다. 각 profile의 centered residual에 AR(1)을 적합하고, OU innovation의 자기상관과 비정규 tail을 진단한다.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_residual_diagnostics.py')],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

## 2. Train-only 구조 선별 결과

In [ ]:
decision = json.loads((OUTPUT / 'residual_structure_decision.json').read_text(encoding='utf-8'))
display(decision)
aggregate = pd.read_csv(OUTPUT / 'aggregate_residual_diagnostics.csv')
display(aggregate.round(4))

## 3. Residual과 OU innovation

왼쪽은 M3 평균 운전점에서 벗어난 residual, 오른쪽은 OU/AR(1)이 설명하고 남은 innovation이다. 붉은 점은 GMM 진단에서 중앙 background component보다 tail component일 posterior가 0.8 이상인 표본이다.

In [ ]:
display(Image(filename=str(OUTPUT / 'residual_innovation_representative.png')))

## 4. Request rate별 구조 진단

- `phi`: residual의 단기 지속성으로, 값이 충분히 크면 R2 OU가 필요하다.
- `tail ratio`: Gaussian에서 기대되는 3-sigma 초과빈도 대비 실제 innovation의 초과빈도다.
- `BIC 차이`: 양수가 클수록 단일 Gaussian보다 3-component mixture가 innovation 분포를 더 잘 설명한다.

In [ ]:
display(Image(filename=str(OUTPUT / 'train_structure_diagnostic_map.png')))
display(Image(filename=str(OUTPUT / 'innovation_normal_qq.png')))

## 5. 자동 생성 보고서

In [ ]:
display(Markdown((OUTPUT / 'residual_diagnostics_report_ko.md').read_text(encoding='utf-8')))